# EV Charging Forward Forecast Demo

This notebook demonstrates short-horizon recursive forecasting for EV charging demand using the final selected Gradient Boosting models.

Targets:
- **session_count**: hourly number of charging sessions
- **total_kwh**: hourly total charging energy demand

Forecast setup:
- The model uses the **latest observed historical window** as seed input
- Future values are generated **recursively**, so each new prediction is fed back into the history for the next step
- The selected future timestamp is used to provide calendar context such as hour of day, day of week, and month

Important note:
- This is a **forward forecast demo / scenario-based forecast**
- Since the historical dataset ends earlier than the displayed future dates, the forecast should be interpreted as a **model-based future demand scenario**, not as verified ground-truth demand for those future dates

In [128]:
# Final presentation notebook imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from ipywidgets import interact, Dropdown, IntSlider

plt.rcParams["figure.figsize"] = (12, 5)

In [129]:
# Load processed datasets

train_features = pd.read_csv("../data/processed/train_features.csv")
hourly_df = pd.read_csv("../data/processed/hourly_demand.csv")

train_features["hour"] = pd.to_datetime(train_features["hour"], utc=True)
hourly_df["hour"] = pd.to_datetime(hourly_df["hour"], utc=True)

print("Loaded datasets:")
print(f"  train_features shape: {train_features.shape}")
print(f"  hourly_df shape:      {hourly_df.shape}")
print()
print("Observed data range:")
print(f"  start: {hourly_df['hour'].min()}")
print(f"  end:   {hourly_df['hour'].max()}")
print()
print("Latest observed rows:")
display(hourly_df.tail())

Loaded datasets:
  train_features shape: (20997, 17)
  hourly_df shape:      (23687, 7)

Observed data range:
  start: 2019-01-01 03:00:00+00:00
  end:   2021-09-14 01:00:00+00:00

Latest observed rows:


,hour,session_count,total_kwh,hour_of_day,day_of_week,month,is_weekend
23682,2021-09-13 21:00:00+00:00,2,9.000,21,0,9,0
23683,2021-09-13 22:00:00+00:00,1,17.720,22,0,9,0
23684,2021-09-13 23:00:00+00:00,1,2.018,23,0,9,0
23685,2021-09-14 00:00:00+00:00,0,0.000,0,1,9,0
23686,2021-09-14 01:00:00+00:00,1,45.064,1,1,9,0


In [130]:
# Final feature sets

session_features = [
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend",
    "session_lag_1",
    "session_lag_2",
    "session_lag_24",
    "session_lag_168",
    "session_rolling_mean_24",
]

kwh_features = [
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend",
    "session_lag_1",
    "session_lag_2",
    "session_lag_24",
    "session_lag_168",
    "session_rolling_mean_24",
    "kwh_lag_1",
    "kwh_lag_2",
    "kwh_lag_24",
    "kwh_lag_168",
    "kwh_rolling_mean_24",
]

print("Session model features:")
print(session_features)
print()
print("kWh model features:")
print(kwh_features)

Session model features:
['hour_of_day', 'day_of_week', 'month', 'is_weekend', 'session_lag_1', 'session_lag_2', 'session_lag_24', 'session_lag_168', 'session_rolling_mean_24']

kWh model features:
['hour_of_day', 'day_of_week', 'month', 'is_weekend', 'session_lag_1', 'session_lag_2', 'session_lag_24', 'session_lag_168', 'session_rolling_mean_24', 'kwh_lag_1', 'kwh_lag_2', 'kwh_lag_24', 'kwh_lag_168', 'kwh_rolling_mean_24']


In [131]:
# Train final Gradient Boosting models for forward forecasting

X_session = train_features[session_features]
y_session = train_features["session_count"]

X_kwh = train_features[kwh_features]
y_kwh = train_features["total_kwh"]

session_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

kwh_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

session_model.fit(X_session, y_session)
kwh_model.fit(X_kwh, y_kwh)

print("Final models trained successfully.")
print("  Session model: Gradient Boosting Regressor")
print("  kWh model:     Gradient Boosting Regressor")

Final models trained successfully.
  Session model: Gradient Boosting Regressor
  kWh model:     Gradient Boosting Regressor


In [132]:
def build_session_feature_row(timestamp: pd.Timestamp, history_df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "hour_of_day": [timestamp.hour],
        "day_of_week": [timestamp.dayofweek],
        "month": [timestamp.month],
        "is_weekend": [1 if timestamp.dayofweek in [5, 6] else 0],
        "session_lag_1": [history_df["session_count"].iloc[-1]],
        "session_lag_2": [history_df["session_count"].iloc[-2]],
        "session_lag_24": [history_df["session_count"].iloc[-24]],
        "session_lag_168": [history_df["session_count"].iloc[-168]],
        "session_rolling_mean_24": [history_df["session_count"].iloc[-24:].mean()],
    })


def build_kwh_feature_row(timestamp: pd.Timestamp, history_df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "hour_of_day": [timestamp.hour],
        "day_of_week": [timestamp.dayofweek],
        "month": [timestamp.month],
        "is_weekend": [1 if timestamp.dayofweek in [5, 6] else 0],
        "session_lag_1": [history_df["session_count"].iloc[-1]],
        "session_lag_2": [history_df["session_count"].iloc[-2]],
        "session_lag_24": [history_df["session_count"].iloc[-24]],
        "session_lag_168": [history_df["session_count"].iloc[-168]],
        "session_rolling_mean_24": [history_df["session_count"].iloc[-24:].mean()],
        "kwh_lag_1": [history_df["total_kwh"].iloc[-1]],
        "kwh_lag_2": [history_df["total_kwh"].iloc[-2]],
        "kwh_lag_24": [history_df["total_kwh"].iloc[-24]],
        "kwh_lag_168": [history_df["total_kwh"].iloc[-168]],
        "kwh_rolling_mean_24": [history_df["total_kwh"].iloc[-24:].mean()],
    })

In [133]:
def recursive_forecast(history_df: pd.DataFrame, forecast_start: pd.Timestamp, horizon_hours: int) -> pd.DataFrame:
    history = history_df.copy().reset_index(drop=True)
    forecast_rows = []

    for step in range(horizon_hours):
        current_time = forecast_start + pd.Timedelta(hours=step)

        X_session = build_session_feature_row(current_time, history)
        pred_session = session_model.predict(X_session)[0]

        X_kwh = build_kwh_feature_row(current_time, history)
        pred_kwh = kwh_model.predict(X_kwh)[0]

        forecast_rows.append({
            "hour": current_time,
            "session_count": pred_session,
            "total_kwh": pred_kwh,
            "day_name": current_time.day_name(),
            "hour_of_day": current_time.hour,
        })

        new_row = pd.DataFrame([{
            "hour": current_time,
            "session_count": pred_session,
            "total_kwh": pred_kwh,
        }])

        history = pd.concat([history, new_row], ignore_index=True)

    return pd.DataFrame(forecast_rows)

In [134]:
# Presentation-oriented forecast start options
# These timestamps provide future calendar contexts for the forward forecast demo.

start_options = [
    "2026-03-30 00:00:00+00:00",
    "2026-03-31 00:00:00+00:00",
    "2026-04-01 00:00:00+00:00",
]

In [135]:
import pandas as pd

def compact_forecast_table(df: pd.DataFrame, head_rows: int = 3, tail_rows: int = 3) -> pd.DataFrame:
    """
    Return a compact presentation table showing the first few and last few rows.
    """
    if len(df) <= head_rows + tail_rows:
        return df.copy()

    head = df.head(head_rows)
    tail = df.tail(tail_rows)

    ellipsis_row = pd.DataFrame(
        {col: ["..."] for col in df.columns},
        index=["..."]
    )

    compact_df = pd.concat([head, ellipsis_row, tail], axis=0)
    return compact_df

In [136]:
def forecast_view(future_start, horizon_hours, history_window):
    forecast_start = pd.Timestamp(future_start)

    if history_window < 168:
        raise ValueError("history_window must be at least 168 because the models use lag_168 features.")

    seed_history = hourly_df.iloc[-history_window:].copy()

    forecast_df = recursive_forecast(
        history_df=seed_history,
        forecast_start=forecast_start,
        horizon_hours=horizon_hours
    )

    # Prevent negative forecasts
    forecast_df["session_count"] = forecast_df["session_count"].clip(lower=0)
    forecast_df["total_kwh"] = forecast_df["total_kwh"].clip(lower=0)

    # Round for cleaner presentation
    forecast_df_display = forecast_df.copy()
    forecast_df_display["session_count"] = forecast_df_display["session_count"].round(2)
    forecast_df_display["total_kwh"] = forecast_df_display["total_kwh"].round(2)

    print("Forward forecast demo")
    print(f"Forecast start:    {forecast_df_display['hour'].iloc[0]}")
    print(f"Forecast end:      {forecast_df_display['hour'].iloc[-1]}")
    print(f"Forecast horizon:  {horizon_hours} hours")
    print(f"Seed history:      latest {history_window} observed hours")
    print()

    forecast_table_display = compact_forecast_table(
        forecast_df_display[["hour", "session_count", "total_kwh", "day_name", "hour_of_day"]],
        head_rows=3,
        tail_rows=3,
    )
    display(forecast_table_display)

    # Plot session forecast
    plt.figure(figsize=(10, 4))
    plt.plot(
        range(-history_window, 0),
        seed_history["session_count"].values,
        label="Observed History"
    )
    plt.plot(
        range(0, horizon_hours),
        forecast_df["session_count"].values,
        label="Forecast"
    )
    plt.axvline(x=0, linestyle="--")
    plt.title(f"Forward Forecast for Session Count Starting {forecast_start}")
    plt.xlabel("Hours from Forecast Start")
    plt.ylabel("Session Count")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Plot kWh forecast
    plt.figure(figsize=(10, 4))
    plt.plot(
        range(-history_window, 0),
        seed_history["total_kwh"].values,
        label="Observed History"
    )
    plt.plot(
        range(0, horizon_hours),
        forecast_df["total_kwh"].values,
        label="Forecast"
    )
    plt.axvline(x=0, linestyle="--")
    plt.title(f"Forward Forecast for Total kWh Starting {forecast_start}")
    plt.xlabel("Hours from Forecast Start")
    plt.ylabel("Total kWh")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
interact(
    forecast_view,
    future_start=Dropdown(
        options=start_options,
        value=start_options[0],
        description="Start"
    ),
    horizon_hours=IntSlider(
        value=24,
        min=6,
        max=96,
        step=6,
        description="Hours"
    ),
    history_window=IntSlider(
        value=168,
        min=168,
        max=336,
        step=24,
        description="History"
    )
);

interactive(children=(Dropdown(description='Start', options=('2026-03-30 00:00:00+00:00', '2026-03-31 00:00:00…